In [1]:
from pathlib import Path
import os
import duckdb
import pandas as pd
import numpy as np
import sklearn

# Find the project root.
ROOT = (
    Path.cwd()
    if (Path.cwd() / "data").exists()
    else Path.cwd().parent
)
os.chdir(ROOT)

# Load the compact Parquet dataset.
sample_path = ROOT / "data" / "stage4_model_sample.parquet"

assert sample_path.exists(), (
    f"Dataset not found: {sample_path}"
)

con = duckdb.connect()

df = con.execute(
    f"SELECT * FROM read_parquet('{sample_path.as_posix()}')"
).df()

print("Project root:", ROOT)
print("Dataset shape:", df.shape)
print("Scikit-learn version:", sklearn.__version__)

display(df.head())

display(
    df.groupby("split").agg(
        rows=("target_long_wait", "size"),
        long_waits=("target_long_wait", "sum"),
        long_wait_rate=("target_long_wait", "mean")
    ).assign(
        long_wait_pct=lambda x: 100 * x["long_wait_rate"]
    ).drop(columns="long_wait_rate")
)

Project root: /Users/waheed/Desktop/Project1
Dataset shape: (201602, 9)
Scikit-learn version: 1.7.1


,split,request_date,request_hour,request_dow,pickup_zone,company,shared_requested,wav_requested,target_long_wait
0,train,2025-01-01,0,3,239,Uber,N,N,0
1,train,2025-01-01,0,3,239,Uber,N,N,0
2,train,2025-01-01,0,3,36,Lyft,N,N,0
3,train,2025-01-01,0,3,37,Uber,N,N,0
4,train,2025-01-01,0,3,213,Lyft,N,N,0


,rows,long_waits,long_wait_pct
split,,,
test,46443,2107,4.536744
train,134168,7468,5.566156
validation,20991,1041,4.959268


In [2]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)
import math

# 1. Separate training and validation data.
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()

target = "target_long_wait"
keys = ["pickup_zone", "request_hour"]

# 2. Estimate the overall training long-wait rate.
global_rate = train_df[target].mean()

# 3. Calculate historical long-wait rates for each zone-hour.
zone_hour_rates = (
    train_df
    .groupby(keys)[target]
    .agg(n_trips="size", long_waits="sum")
    .reset_index()
)

# 4. Smooth small-group estimates toward the overall rate.
# A value of 100 acts like 100 prior trips at the global rate.
alpha = 100

zone_hour_rates["baseline_probability"] = (
    zone_hour_rates["long_waits"] + alpha * global_rate
) / (
    zone_hour_rates["n_trips"] + alpha
)

# 5. Assign historical probabilities to validation trips.
val_scored = val_df.merge(
    zone_hour_rates[
        keys + ["n_trips", "baseline_probability"]
    ],
    on=keys,
    how="left",
    validate="many_to_one"
)

# For zone-hour combinations absent from training, use the
# overall training rate instead.
unseen_combinations = val_scored["baseline_probability"].isna().sum()

val_scored["baseline_probability"] = (
    val_scored["baseline_probability"].fillna(global_rate)
)

y_true = val_scored[target].astype(int)
y_prob = val_scored["baseline_probability"]

# 6. Evaluate on validation data.
top_k = math.ceil(len(val_scored) * 0.10)
top_risk = val_scored.nlargest(top_k, "baseline_probability")

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print(f"Training long-wait rate: {global_rate:.2%}")
print("Unseen zone-hour combinations:", unseen_combinations)

print("\nBASELINE VALIDATION RESULTS")
print(f"Average precision: {average_precision_score(y_true, y_prob):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"Brier score: {brier_score_loss(y_true, y_prob):.4f}")

print("\nTOP 10% HIGHEST-RISK VALIDATION TRIPS")
print(f"Trips flagged: {top_k}")
print(f"Long-wait rate in flagged trips: {top_risk[target].mean():.2%}")
print(
    "Share of all validation long waits captured: "
    f"{top_risk[target].sum() / y_true.sum():.2%}"
)

Training rows: 134168
Validation rows: 20991
Training long-wait rate: 5.57%
Unseen zone-hour combinations: 27

BASELINE VALIDATION RESULTS
Average precision: 0.1212
ROC-AUC: 0.6725
Brier score: 0.0460

TOP 10% HIGHEST-RISK VALIDATION TRIPS
Trips flagged: 2100
Long-wait rate in flagged trips: 14.62%
Share of all validation long waits captured: 29.49%


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

import numpy as np
import pandas as pd

# Features available at request time.
features = [
    "pickup_zone",
    "request_hour",
    "request_dow",
    "company",
    "shared_requested",
    "wav_requested",
]

# Prepare train and validation features.
X_train = train_df[features].fillna("Unknown").astype(str)
y_train = train_df["target_long_wait"].astype(int)

X_val = val_df[features].fillna("Unknown").astype(str)
y_val = val_df["target_long_wait"].astype(int)

# Encode categories; ignore categories unseen during training.
preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        features
    )
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            C=1.0,
            class_weight=None,
            random_state=42
        )
    )
])

# Fit only on the training period.
logistic_model.fit(X_train, y_train)

# Predict validation probabilities.
lr_probs = logistic_model.predict_proba(X_val)[:, 1]

# Recreate baseline probabilities in validation-row order.
val_keys = pd.MultiIndex.from_frame(val_df[keys])

baseline_probs = (
    zone_hour_rates
    .set_index(keys)["baseline_probability"]
    .reindex(val_keys)
    .to_numpy(dtype=float)
)

baseline_probs = np.where(
    np.isnan(baseline_probs),
    global_rate,
    baseline_probs
)

# Evaluate both approaches on the same validation rows.
def evaluate_model(name, actual, probabilities):
    results = {
        "model": name,
        "average_precision": average_precision_score(
            actual, probabilities
        ),
        "roc_auc": roc_auc_score(actual, probabilities),
        "brier_score": brier_score_loss(actual, probabilities),
    }

    top_k = int(np.ceil(len(actual) * 0.10))
    top_idx = np.argsort(-probabilities, kind="stable")[:top_k]

    results["top10_long_wait_rate_pct"] = (
        actual.iloc[top_idx].mean() * 100
    )

    results["long_wait_capture_pct"] = (
        actual.iloc[top_idx].sum() / actual.sum() * 100
    )

    return results

comparison = pd.DataFrame([
    evaluate_model("Zone-hour baseline", y_val, baseline_probs),
    evaluate_model("Logistic regression", y_val, lr_probs),
])

display(comparison.round(4))

,model,average_precision,roc_auc,brier_score,top10_long_wait_rate_pct,long_wait_capture_pct
0,Zone-hour baseline,0.1212,0.6725,0.0460,14.6190,29.4909
1,Logistic regression,0.1317,0.7008,0.0456,16.6667,33.6215


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Keep memory and runtime manageable.
rf_preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        features
    )
])

rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=16,
            min_samples_leaf=50,
            max_features="sqrt",
            n_jobs=2,
            random_state=42
        )
    )
])

# Train only on the training period.
rf_model.fit(X_train, y_train)

# Evaluate on validation data; the final test set remains untouched.
rf_probs = rf_model.predict_proba(X_val)[:, 1]

comparison = pd.DataFrame([
    evaluate_model("Zone-hour baseline", y_val, baseline_probs),
    evaluate_model("Logistic regression", y_val, lr_probs),
    evaluate_model("Random forest", y_val, rf_probs),
])

display(comparison.round(4))

,model,average_precision,roc_auc,brier_score,top10_long_wait_rate_pct,long_wait_capture_pct
0,Zone-hour baseline,0.1212,0.6725,0.0460,14.6190,29.4909
1,Logistic regression,0.1317,0.7008,0.0456,16.6667,33.6215
2,Random forest,0.1367,0.6910,0.0454,17.2381,34.7743


In [5]:
# Final evaluation on the untouched test period.
test_df = df[df["split"] == "test"].copy()

X_test = test_df[features].fillna("Unknown").astype(str)
y_test = test_df["target_long_wait"].astype(int)

# Baseline probabilities are derived exclusively from training data.
test_keys = pd.MultiIndex.from_frame(test_df[keys])

test_baseline_probs = (
    zone_hour_rates
    .set_index(keys)["baseline_probability"]
    .reindex(test_keys)
    .to_numpy(dtype=float)
)

test_baseline_probs = np.where(
    np.isnan(test_baseline_probs),
    global_rate,
    test_baseline_probs
)

# Models are already fitted using training data only.
lr_test_probs = logistic_model.predict_proba(X_test)[:, 1]
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]

test_comparison = pd.DataFrame([
    evaluate_model(
        "Zone-hour baseline", y_test, test_baseline_probs
    ),
    evaluate_model(
        "Logistic regression", y_test, lr_test_probs
    ),
    evaluate_model(
        "Random forest", y_test, rf_test_probs
    ),
])

print("TEST PERIOD:", test_df["request_date"].min(),
      "to", test_df["request_date"].max())
print("Test rows:", len(test_df))
print("Actual long-wait rate:", round(100 * y_test.mean(), 2), "%")

display(test_comparison.round(4))

TEST PERIOD: 2025-01-25 00:00:00 to 2025-01-31 00:00:00
Test rows: 46443
Actual long-wait rate: 4.54 %


,model,average_precision,roc_auc,brier_score,top10_long_wait_rate_pct,long_wait_capture_pct
0,Zone-hour baseline,0.1204,0.6643,0.0421,12.6803,27.9544
1,Logistic regression,0.1282,0.7015,0.0419,15.6297,34.4566
2,Random forest,0.1358,0.6905,0.0416,16.2110,35.7380


In [6]:
# Preserve validation and final test metrics in one file.
validation_results = comparison.copy()
validation_results["evaluation_split"] = "validation"

test_results = test_comparison.copy()
test_results["evaluation_split"] = "test"

model_results = pd.concat(
    [validation_results, test_results],
    ignore_index=True
)

# Put the evaluation split before the model name.
cols = ["evaluation_split"] + [
    c for c in model_results.columns
    if c != "evaluation_split"
]
model_results = model_results[cols]

output_path = ROOT / "data" / "stage4_model_performance.csv"
model_results.to_csv(output_path, index=False)

display(model_results.round(4))
print("Saved:", output_path)

,evaluation_split,model,average_precision,roc_auc,brier_score,top10_long_wait_rate_pct,long_wait_capture_pct
0,validation,Zone-hour baseline,0.1212,0.6725,0.0460,14.6190,29.4909
1,validation,Logistic regression,0.1317,0.7008,0.0456,16.6667,33.6215
2,validation,Random forest,0.1367,0.6910,0.0454,17.2381,34.7743
3,test,Zone-hour baseline,0.1204,0.6643,0.0421,12.6803,27.9544
4,test,Logistic regression,0.1282,0.7015,0.0419,15.6297,34.4566
5,test,Random forest,0.1358,0.6905,0.0416,16.2110,35.7380


Saved: /Users/waheed/Desktop/Project1/data/stage4_model_performance.csv


In [7]:
from pathlib import Path
import pandas as pd

csv_files = sorted(
    p for p in ROOT.rglob("*.csv")
    if not any(
        part in {".git", ".venv", "venv", "__pycache__"}
        for part in p.parts
    )
)

print("CSV files found:", len(csv_files))

for path in csv_files:
    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()
        print(f"\n{path.relative_to(ROOT)}")
        print("Columns:", columns)
    except Exception as exc:
        print(f"\nCould not inspect {path.name}: {exc}")

CSV files found: 52

dashboard/data/.ipynb_checkpoints/2a_wait_bins_by_company-checkpoint.csv
Columns: ['hvfhs_license_num', 'company', 'bin_start_min', 'bin_label', 'n_rows', 'pct_of_scoped', 'pct_of_valid', 'cum_pct_of_valid']

dashboard/data/.ipynb_checkpoints/2f_fare_component_audit-checkpoint.csv
Columns: ['company', 'n_rows', 'n_base_fare_positive', 'pct_rows_tolls', 'pct_rows_bcf', 'pct_rows_sales_tax', 'pct_rows_congestion_surcharge', 'pct_rows_airport_fee', 'pct_rows_cbd_fee', 'pct_rows_tips', 'sales_tax_to_base_p25', 'sales_tax_to_base_p50', 'sales_tax_to_base_p75', 'bcf_to_base_p25', 'bcf_to_base_p50', 'bcf_to_base_p75']

dashboard/data/.ipynb_checkpoints/2f_pay_by_hour_company-checkpoint.csv
Columns: ['company', 'request_hour', 'n', 'pay_p50', 'ppm_p50', 'ppmin_p50', 'median_miles']

dashboard/data/.ipynb_checkpoints/2f_pay_by_wait_band-checkpoint.csv
Columns: ['company', 'wait_band', 'n', 'pct_of_company_valid', 'pay_p25', 'pay_p50', 'pay_p75', 'ppm_p50', 'ppmin_p50', 'med

In [8]:
pay = pd.read_csv(ROOT / "dashboard/data/2f_pay_distribution.csv")
ppm = pd.read_csv(ROOT / "dashboard/data/2f_ppm_distribution.csv")
ppmin = pd.read_csv(ROOT / "dashboard/data/2f_ppmin_distribution.csv")

companies = ["Uber", "Lyft"]

pay_kpi = pay[pay["company"].isin(companies)][["company", "n", "p50"]].rename(
    columns={"n": "pay_rows", "p50": "median_pay_per_trip"}
)
ppm_kpi = ppm[ppm["company"].isin(companies)][["company", "n_eligible", "p50"]].rename(
    columns={"n_eligible": "ppm_rows", "p50": "median_pay_per_mile"}
)
ppmin_kpi = ppmin[ppmin["company"].isin(companies)][["company", "n_eligible", "p50"]].rename(
    columns={"n_eligible": "ppmin_rows", "p50": "median_pay_per_minute"}
)

pay_kpis = pay_kpi.merge(ppm_kpi, on="company").merge(ppmin_kpi, on="company")
display(pay_kpis)


,company,pay_rows,median_pay_per_trip,ppm_rows,median_pay_per_mile,ppmin_rows,median_pay_per_minute
0,Uber,15356455,14.16,15354057,4.596,15356451,0.929
1,Lyft,5049211,13.32,5049003,4.731,5049211,0.893
